# Advanced Dynamic Programming

The **dynamic programming** notebook established the two ingredients — *optimal substructure* and *overlapping subproblems* — and worked the classic grid DPs (coin change, LCS, edit distance, knapsack), where the state was an index or a pair of positions. This sequel is about the cases where **the state stops being a simple index**: it's an interval `(i, j)`, a *subset* of items packed into a bitmask, a position in a tree, a prefix of a number's digits, a finite-state-machine mode, or a probability distribution over outcomes.

Every category below follows the same drill: **name the pattern → state the design (what the state *is*) → a canonical worked problem, coded and *proven* by an `assert` against a brute-force oracle → a recognition cue.** Each DP is fact-checked against a brute force (permutations, plain recursion, or Monte-Carlo simulation) over seeded-random small inputs — if the clever table and the dumb oracle ever disagreed, the cell would raise.

**Contents**

1. **Beyond the basics** — what changes when the state isn't an index
2. **Interval DP** — matrix-chain multiplication
3. **Bitmask DP (DP over subsets)** — Held-Karp travelling salesman
4. **Tree DP** — maximum-weight independent set on a tree
5. **Digit DP** — counting numbers with a digit property in `[0, N]`
6. **State-machine DP** — buy/sell stock with a cooldown
7. **Expected-value / probability DP** — absorbing Markov chain
8. **How to recognize each + cheat-sheet**

## 1. Beyond the basics — what changes when the state isn't an index

A DP is fully specified by three things: the **state** (the parameters that identify a subproblem — this is your cache key or your table's axes), the **transition** (how a state's answer is built from *smaller* states), and the **base cases**. In the **dynamic programming** notebook the state was always a position in a sequence: `amount`, or a pair `(i, j)` of string indices. The recurrence and the code followed mechanically once that state was pinned down.

The "advanced" families are advanced only in the *shape of the state*. The transition machinery is the same. Here is the whole tour at a glance:

| Family | State *is*... | Transition orders over... | Typical cost |
|---|---|---|---|
| **Interval DP** | a subarray `(i, j)` | the split point `k` inside `(i, j)` | O(n³) |
| **Bitmask DP** | a *subset* `mask` (+ maybe a position) | which element to add/remove next | O(2ⁿ · n²) |
| **Tree DP** | a node (+ a small flag) | a node's children, post-order | O(n) |
| **Digit DP** | digit position + `(tight, started, ...)` | the next digit `0..9` | O(digits · states · 10) |
| **State-machine DP** | day + which FSM state you're in | the legal state transitions | O(n · states) |
| **Probability DP** | an outcome / configuration | weighted by transition probabilities | depends |

The recurring difficulty is the same one the basics notebook flagged: **defining the state**. Get that right and order the transitions so every state is solved before it's needed, and the rest is bookkeeping. We prove each one against a brute force so there's no hand-waving.

In [1]:
import random
random.seed(7)          # one seed for the whole notebook; every oracle test is reproducible
from functools import lru_cache
from itertools import permutations, combinations

print("seeded. python", __import__('sys').version.split()[0])

seeded. python 3.14.3


## 2. Interval DP — matrix-chain multiplication

**Pattern.** The state is a *contiguous interval* `(i, j)` of the input, and you solve it by trying every way to **split** the interval at some `k` with `i ≤ k < j`, combining the answers for `(i, k)` and `(k+1, j)` plus a merge cost. Short intervals are solved first, so you fill the table by **increasing interval length**.

**Design.** `dp[i][j]` = optimal cost to handle the sub-chain from `i` to `j`. Base case: a length-1 interval costs 0. Transition: `dp[i][j] = min over k of dp[i][k] + dp[k+1][j] + cost(i, k, j)`.

**Canonical problem — matrix-chain multiplication.** Given matrices with dimensions `p[0]×p[1], p[1]×p[2], …`, multiplication is associative, so the *answer* is fixed but the *cost* depends on parenthesization. Multiplying an `a×b` by a `b×c` matrix costs `a·b·c` scalar multiplications. We want the cheapest order. Here `cost(i, k, j) = p[i]·p[k+1]·p[j+1]`.

We prove the O(n³) table against a brute-force recursion that explicitly tries *every* split with no memoization.

In [2]:
def matrix_chain_dp(p):
    # p has len n+1 for n matrices; dp[i][j] = min cost to multiply matrices i..j
    n = len(p) - 1
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):                 # interval length, shortest first
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = min(
                dp[i][k] + dp[k + 1][j] + p[i] * p[k + 1] * p[j + 1]
                for k in range(i, j)               # every split point
            )
    return dp[0][n - 1]

def matrix_chain_brute(p):
    n = len(p) - 1
    def best(i, j):                                # no memo: pure exponential oracle
        if i == j:
            return 0
        return min(best(i, k) + best(k + 1, j) + p[i] * p[k + 1] * p[j + 1]
                   for k in range(i, j))
    return best(0, n - 1)

# textbook instance: dims 40x20, 20x30, 30x10, 10x30  ->  optimal 26000
demo = [40, 20, 30, 10, 30]
print("matrix-chain min scalar mults:", matrix_chain_dp(demo))

# PROVE: table == brute force over many seeded-random chains
for _ in range(300):
    p = [random.randint(1, 12) for _ in range(random.randint(2, 7))]
    assert matrix_chain_dp(p) == matrix_chain_brute(p)
print("verified: interval DP == brute-force recursion over 300 random chains")

matrix-chain min scalar mults: 26000
verified: interval DP == brute-force recursion over 300 random chains


**Recognition cue.** You're optimizing over a *contiguous range*, the answer for a range is built by **choosing a split / pivot inside it**, and merging two adjacent sub-ranges has a cost. Burst balloons, min-cost to merge stones, optimal BST, and "remove boxes" are all this shape. The tell is the `dp[i][k] + dp[k+1][j]` join over an inner `k`.

## 3. Bitmask DP (DP over subsets) — Held-Karp travelling salesman

**Pattern.** When `n` is small (≤ ~20) and a subproblem is identified by **which subset of elements you've used**, encode that subset as the bits of an integer — a *bitmask*. Subsets become array indices, set-union/intersection become `|`/`&`, "is element `i` in the set?" becomes `mask >> i & 1`. (The **bit manipulation** notebook covers these tricks in full; here they're the substrate for the state.)

**Design.** State = `(mask, i)` = "I've visited exactly the cities in `mask`, and I'm currently standing at city `i`". Transition: extend to an unvisited city `j` for `dist[i][j]`.

**Canonical problem — travelling salesman (Held-Karp).** Brute force is `(n-1)!` permutations; Held-Karp is `O(2ⁿ · n²)` — still exponential, but it makes `n = 13` instant where `13! ≈ 6×10⁹` permutations would not. We keep `n ≤ 10` so the cell is fast, and prove the DP equals the minimum over *all permutations*.

In [3]:
def tsp_held_karp(dist):
    n = len(dist)
    FULL = (1 << n) - 1
    # dp[mask][i] = min cost of a path that starts at 0, visits exactly `mask`, ends at i
    dp = [[float("inf")] * n for _ in range(1 << n)]
    dp[1][0] = 0                                   # start: only city 0 visited, sitting at 0
    for mask in range(1 << n):
        if not (mask & 1):                         # every tour starts at city 0
            continue
        for i in range(n):
            if dp[mask][i] == float("inf") or not (mask >> i & 1):
                continue
            for j in range(n):
                if mask >> j & 1:                  # j already visited
                    continue
                nm = mask | (1 << j)
                cand = dp[mask][i] + dist[i][j]
                if cand < dp[nm][j]:
                    dp[nm][j] = cand
    return min(dp[FULL][i] + dist[i][0] for i in range(n))   # close the cycle back to 0

def tsp_brute(dist):
    n = len(dist)
    best = float("inf")
    for perm in permutations(range(1, n)):         # fix city 0 as start; try all orders
        tour = (0,) + perm
        cost = sum(dist[tour[k]][tour[k + 1]] for k in range(n - 1)) + dist[tour[-1]][0]
        best = min(best, cost)
    return best

def random_dist(n):
    d = [[0] * n for _ in range(n)]
    for a in range(n):
        for b in range(a + 1, n):
            d[a][b] = d[b][a] = random.randint(1, 30)
    return d

d = random_dist(7)
print("optimal tour cost (n=7):", tsp_held_karp(d))

# PROVE: Held-Karp == min over all (n-1)! permutations, on seeded-random graphs
for n in range(2, 9):                              # n<=8 keeps brute (7! = 5040) fast
    for _ in range(40):
        d = random_dist(n)
        assert tsp_held_karp(d) == tsp_brute(d)
print("verified: Held-Karp == brute-force over permutations, n=2..8")

optimal tour cost (n=7): 26
verified: Held-Karp == brute-force over permutations, n=2..8


**Recognition cue.** Small `n` (a `2ⁿ` factor is affordable) and a subproblem is "given that I've already used *this set* of things, what's best?" — assignment problems, TSP, set cover, counting Hamiltonian paths, "shortest superstring". The tell is `n ≤ ~20` plus a state that is fundamentally a *subset*. See also the subset-enumeration tricks in the **bit manipulation** notebook.

## 4. Tree DP — maximum-weight independent set on a tree

**Pattern.** The state is a **node**, and a node's answer is composed from its **children's** answers. Because a tree has no cycles, a single post-order traversal (children before parent) solves every subproblem exactly once — `O(n)`. Often the state carries a tiny flag for "this node is included / excluded", giving two values per node. (The **tree algorithms** notebook covers traversals and rooting; here a rooted tree is the DP's dependency order.)

**Design.** Root the tree anywhere. For each node `u` return a pair `(take, skip)`:
- `take` = best total *if `u` is in the set* → `weight[u] + Σ skip(child)` (children must be excluded)
- `skip` = best total *if `u` is excluded* → `Σ max(take(child), skip(child))` (children free either way)

**Canonical problem — maximum-weight independent set (MWIS).** Choose a subset of nodes with no two adjacent, maximizing total weight. We prove the linear tree DP against a brute force that checks *every* subset of nodes for independence.

In [4]:
def mwis_tree(n, edges, weight):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)

    # iterative post-order to avoid recursion-limit worries; returns (take, skip) per node
    take = [0] * n
    skip = [0] * n
    parent = [-1] * n
    order, stack, seen = [], [0], {0}
    while stack:                                   # BFS to get a parent array + visit order
        u = stack.pop(); order.append(u)
        for v in adj[u]:
            if v not in seen:
                seen.add(v); parent[v] = u; stack.append(v)
    for u in reversed(order):                       # children finalised before parent
        take[u] = weight[u]
        for v in adj[u]:
            if v != parent[u]:
                take[u] += skip[v]
                skip[u] += max(take[v], skip[v])
    return max(take[0], skip[0])

def mwis_brute(n, edges, weight):
    eset = {frozenset(e) for e in edges}
    best = 0
    for r in range(n + 1):
        for subset in combinations(range(n), r):
            s = set(subset)
            if all(frozenset((a, b)) not in eset                 # independent?
                   for a, b in combinations(subset, 2)):
                best = max(best, sum(weight[i] for i in s))
    return best

def random_tree(n):
    edges = [(random.randint(0, i - 1), i) for i in range(1, n)]   # each node attaches to an earlier one
    w = [random.randint(1, 20) for _ in range(n)]
    return n, edges, w

n, edges, w = random_tree(8)
print("max-weight independent set:", mwis_tree(n, edges, w))

# PROVE: tree DP == brute-force-over-all-subsets, on seeded-random trees
for size in range(1, 13):
    for _ in range(25):
        n, edges, w = random_tree(size)
        assert mwis_tree(n, edges, w) == mwis_brute(n, edges, w)
print("verified: tree DP == brute force over all subsets, n=1..12")

max-weight independent set: 61


verified: tree DP == brute force over all subsets, n=1..12


**Recognition cue.** The structure is a **tree** (or any DAG you can root) and a node's optimum depends only on its subtrees, usually with a small per-node flag (`include` / `exclude`, or "k still to delete"). Tree knapsack, tree diameter, counting matchings, and "distribute coins in a binary tree" all fit. The tell is *post-order composition of children's answers*.

## 5. Digit DP — counting numbers with a digit property in `[0, N]`

**Pattern.** To *count* (or sum over) all integers in `[0, N]` satisfying some digit constraint, build the number **one digit at a time, most-significant first**, and carry just enough state to decide the next digit. The crucial flag is **`tight`**: are we still hugging the prefix of `N` (so the next digit is capped at `N`'s digit) or have we already gone strictly below (so the next digit is free, `0..9`)? A second flag **`started`** handles leading zeros.

**Design.** State = `(pos, tight, started, feature)` where `feature` is whatever the property needs (here: the previous digit, to forbid equal adjacent digits). Memoize over the *position-independent* part `(pos, prev, started)` only when `tight` is false — tight states are unique per prefix and shouldn't be cached.

**Canonical problem — count integers in `[0, N]` with no two equal adjacent digits.** We prove the digit DP against a literal loop that checks every integer from 0 to N.

In [5]:
def count_no_equal_adjacent(N):
    if N < 0:
        return 0
    digits = [int(c) for c in str(N)]
    n = len(digits)

    @lru_cache(maxsize=None)
    def go(pos, prev, started, tight):
        # prev = last placed digit (10 = none yet); started = have we placed a nonzero digit
        if pos == n:
            return 1                               # reached a full valid number (incl. 0)
        limit = digits[pos] if tight else 9
        total = 0
        for d in range(0, limit + 1):
            if started and d == prev:              # equal adjacent digits -> forbidden
                continue
            nstarted = started or d != 0
            nprev = d if nstarted else 10          # leading zeros don't count as 'prev'
            total += go(pos + 1, nprev, nstarted, tight and d == limit)
        return total

    result = go(0, 10, False, True)
    go.cache_clear()                               # tight states leak across N; clear between calls
    return result

def brute_no_equal_adjacent(N):
    def ok(x):
        s = str(x)
        return all(s[i] != s[i + 1] for i in range(len(s) - 1))
    return sum(1 for x in range(N + 1) if ok(x))

print("count in [0, 1000] with no two equal adjacent digits:", count_no_equal_adjacent(1000))

# PROVE: digit DP == literal per-integer count, on seeded-random N
for _ in range(200):
    N = random.randint(0, 5000)
    assert count_no_equal_adjacent(N) == brute_no_equal_adjacent(N)
print("verified: digit DP == brute-force counting over 200 random N in [0, 5000]")

count in [0, 1000] with no two equal adjacent digits: 820


verified: digit DP == brute-force counting over 200 random N in [0, 5000]


**Recognition cue.** You're **counting / summing over a numeric range** `[0, N]` (or `[L, R]` via `f(R) − f(L−1)`) under a per-digit rule, and `N` is far too large to iterate. The signature state is `(position, tight, started, …)`. "Numbers with digit-sum k", "no `4`/`7` digits", "≤ k distinct digits", and Project-Euler-style digit counts are all digit DP.

## 6. State-machine DP — buy/sell stock with a cooldown

**Pattern.** Model the process as a **finite state machine**: at each step you're in one of a few modes, and only certain transitions between modes are legal. The DP carries one value per `(time, mode)`; you advance day by day, taking the best legal incoming transition into each mode.

**Design.** For *best time to buy & sell stock with cooldown*, three modes after day `i`:
- **hold** — currently holding a share
- **sold** — just sold today (the next day must be a rest — the cooldown)
- **rest** — holding nothing and free to buy

Transitions: `hold = max(stay hold, buy from rest)`; `sold = hold + price` (sell); `rest = max(stay rest, yesterday's sold)`. Answer = `max(sold, rest)` on the last day (never end still holding).

**Canonical problem.** Maximize profit over a price series with unlimited transactions but a one-day cooldown after each sale. We prove the O(n) FSM against a brute force that recursively explores *every* buy/sell/skip decision.

In [6]:
def max_profit_cooldown(prices):
    if not prices:
        return 0
    NEG = float("-inf")
    hold, sold, rest = NEG, 0, 0                   # before day 0 you can't be holding
    for p in prices:
        hold, sold, rest = (
            max(hold, rest - p),                   # keep holding, or buy today (from rest)
            hold + p,                              # sell today
            max(rest, sold),                       # stay resting, or come off cooldown
        )
    return max(sold, rest)                         # never end the run still holding

def max_profit_brute(prices):
    n = len(prices)
    def go(i, holding):                            # explore every decision; cooldown via i+2
        if i >= n:
            return 0
        best = go(i + 1, holding)                  # do nothing today
        if holding:
            best = max(best, prices[i] + go(i + 2, False))   # sell -> cooldown one day
        else:
            best = max(best, -prices[i] + go(i + 1, True))   # buy
        return best
    return go(0, False)

print("max profit [1,2,3,0,2] cooldown:", max_profit_cooldown([1, 2, 3, 0, 2]))   # 3

# PROVE: FSM DP == brute-force decision tree, on seeded-random price series
for _ in range(300):
    prices = [random.randint(0, 12) for _ in range(random.randint(0, 11))]
    assert max_profit_cooldown(prices) == max_profit_brute(prices)
print("verified: state-machine DP == brute-force over all decisions, 300 random series")

max profit [1,2,3,0,2] cooldown: 3
verified: state-machine DP == brute-force over all decisions, 300 random series


**Recognition cue.** The problem is a **sequence of steps** where you're in one of a handful of *modes* and legal moves depend on the current mode (hold/sold/rest, "at most k transactions", "even/odd parity so far", string-matching over a DFA). The tell: you can draw a little state diagram, and the DP is one row per mode advanced over time.

## 7. Expected-value / probability DP — absorbing Markov chain

**Pattern.** The "value" being optimized is an **expectation**. A state's expected value is the probability-weighted average of its successors' values, plus a per-step contribution. When the chain has no cycles you can evaluate it by memoized recursion; with cycles (you can return to a state) you solve the resulting **linear system**.

**Design — expected steps to absorption.** For a Markov chain with *transient* states `T` and *absorbing* states, let `E[s]` = expected number of steps to get absorbed starting from `s`. Then `E[absorbing] = 0` and for transient `s`: `E[s] = 1 + Σ P[s→t] · E[t]`. That's a linear system `(I − Q) E = 1` over the transient states, where `Q` is the transient-to-transient sub-matrix. We solve it with `numpy.linalg.solve` (closed form) and **prove it against a seeded Monte-Carlo simulation** of actually walking the chain.

In [7]:
import numpy as np

def expected_steps_to_absorption(P, transient):
    # P: full row-stochastic transition matrix; `transient` = list of transient state indices
    idx = {s: k for k, s in enumerate(transient)}
    m = len(transient)
    Q = np.array([[P[s][t] for t in transient] for s in transient])   # transient -> transient
    E = np.linalg.solve(np.eye(m) - Q, np.ones(m))                    # (I - Q) E = 1
    return {s: float(E[idx[s]]) for s in transient}

def simulate_steps(P, start, absorbing, trials=20000):
    n = len(P)
    total = 0
    for _ in range(trials):
        s, steps = start, 0
        while s not in absorbing:
            r, cum = random.random(), 0.0
            for t in range(n):                     # sample the next state from row P[s]
                cum += P[s][t]
                if r <= cum:
                    s = t; break
            steps += 1
        total += steps
    return total / trials

# Gambler-ish chain on states 0..4; 0 and 4 absorbing, 1/2/3 transient (p up, q down)
p, q = 0.5, 0.5
P = [[0]*5 for _ in range(5)]
P[0][0] = 1.0; P[4][4] = 1.0                       # absorbing barriers
for s in (1, 2, 3):
    P[s][s + 1] = p; P[s][s - 1] = q
absorbing = {0, 4}
transient = [1, 2, 3]

exact = expected_steps_to_absorption(P, transient)
print("exact expected steps from each state:", {s: round(v, 3) for s, v in exact.items()})
# closed form for symmetric walk on 1..3 with barriers at 0,4: E[k] = k*(4-k)  -> 3,4,3
print("known closed form k*(N-k):       ", {s: s * (4 - s) for s in transient})

# PROVE: linear-system DP ~= Monte-Carlo simulation (seeded), within statistical noise
for s in transient:
    sim = simulate_steps(P, s, absorbing)
    assert abs(sim - exact[s]) < 0.15, (s, sim, exact[s])
print("verified: probability DP ~= seeded Monte-Carlo (|err| < 0.15) for each start state")

exact expected steps from each state: {1: 3.0, 2: 4.0, 3: 3.0}
known closed form k*(N-k):        {1: 3, 2: 4, 3: 3}


verified: probability DP ~= seeded Monte-Carlo (|err| < 0.15) for each start state


**Recognition cue.** The question asks for an **expected** count / cost / probability ("expected number of rolls", "probability of reaching X first"). State = an outcome or configuration; the transition is a *probability-weighted* average instead of a min/max. **Acyclic** ⇒ memoized recursion; **cyclic** ⇒ solve the linear system `(I − Q)E = c`. Dice/coin games, random walks, and absorbing chains are the staples.

## 8. How to recognize each + cheat-sheet

The advanced families differ only in **what the state is**. Match the problem's shape to a row:

| If the problem looks like... | It's | State | Transition |
|---|---|---|---|
| optimize over a **contiguous range**, choosing a split inside it | **Interval DP** | `(i, j)` | `min/max over k of dp[i][k] + dp[k+1][j] + merge` |
| small `n` (≤ ~20), "given the **set** I've used so far…" | **Bitmask DP** | `(mask, i)` | add an unused element: `mask | (1<<j)` |
| a **tree / rootable DAG**, node depends on its subtrees | **Tree DP** | node `(+flag)` | post-order combine of children |
| **count/sum over `[0, N]`** under a per-digit rule | **Digit DP** | `(pos, tight, started, …)` | choose next digit `0..limit` |
| a **sequence of steps** with a few legal **modes** | **State-machine DP** | `(t, mode)` | best legal mode→mode move |
| an **expected** value / probability over random outcomes | **Probability DP** | outcome/config | prob-weighted average of successors |

**Cheat-sheet — state shape and cost per category.**

| Category | Canonical problem | State | Order of evaluation | Time |
|---|---|---|---|:---:|
| Interval | matrix-chain mult. | `(i, j)` | increasing interval length | O(n³) |
| Bitmask | TSP (Held-Karp) | `(mask, i)` | increasing popcount of `mask` | O(2ⁿ · n²) |
| Tree | max-weight indep. set | node + `take/skip` | post-order (children first) | O(n) |
| Digit | count in `[0, N]` | `(pos, tight, started, feat)` | most-significant digit first | O(len · feat · 10) |
| State-machine | stock w/ cooldown | `(day, mode)` | left→right over the sequence | O(n · modes) |
| Probability | absorbing Markov chain | transient state | solve `(I−Q)E = c` (or post-order) | O(t³) solve |

**The universal recipe** (unchanged from the **dynamic programming** notebook): (1) spot it's an optimization/count with overlapping subproblems; (2) **define the state** — the hard part, and the only thing that really differs across these families; (3) write the transition in terms of smaller states + base cases; (4) order the evaluation so dependencies come first (interval length, popcount, post-order, digit position, time, topological); (5) shrink space if each layer reads only the previous one. Across this whole notebook every clever table was checked against a dumb oracle — when you design your own, do the same: a brute force on small inputs is the cheapest correctness proof you'll ever write.